## Creates the **Epidemiological Timeline** for a specific region by using the available data on WNV Cases ##

In [1]:
import pandas as pd

enc = 'utf-8'
dec = 'greek8'

pd.options.display.max_columns = None
pd.options.display.max_rows = 10

In [2]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'
NUTS2_LOWER = 'veneto'

In [3]:
data = pd.read_csv(f'../../data/{NUTS0}_WNV_cases_2008-2019_processed.csv', encoding = enc)
data.head(5)

,onset of symptoms,year,month,day,nuts2,nuts3,lau1
0,2008-08-29,2008,8,29,veneto,rovigo,ficarolo
1,2009-08-20,2009,8,20,veneto,venezia,cona
2,2009-08-23,2009,8,23,veneto,rovigo,villadose
3,2009-09-04,2009,9,4,veneto,rovigo,fratta polesine
4,2009-09-10,2009,9,10,veneto,rovigo,occhiobello


In [4]:
df = data[data['nuts2'].isin([NUTS2_LOWER])].copy()

In [5]:
df.reset_index(drop = True, inplace = True)

In [6]:
df

,onset of symptoms,year,month,day,nuts2,nuts3,lau1
0,2008-08-29,2008,8,29,veneto,rovigo,ficarolo
1,2009-08-20,2009,8,20,veneto,venezia,cona
2,2009-08-23,2009,8,23,veneto,rovigo,villadose
3,2009-09-04,2009,9,4,veneto,rovigo,fratta polesine
4,2009-09-10,2009,9,10,veneto,rovigo,occhiobello
...,...,...,...,...,...,...,...
293,2019-09-20,2019,9,20,veneto,venezia,saint maria da sala
294,2019-09-26,2019,9,26,veneto,padova,vigodarzere
295,2019-09-26,2019,9,26,veneto,venezia,caorle
296,2019-10-05,2019,10,5,veneto,venezia,noventa da piave


In [7]:
wnv_lau1_list = df['lau1'].drop_duplicates().sort_values().tolist()

with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2010-2021_LAU1_Units_Processed.txt', 'w', encoding=enc) as f:
    for item in wnv_lau1_list:
        f.write("%s\n" % item)
  
print(f"Number of LAU1 Units in {NUTS2} (from cases): {len(wnv_lau1_list)}")

Number of LAU1 Units in Veneto (from cases): 175


In [8]:
df['period'] = df.apply(lambda row: 1 if (row['day']>=1 and row['day']<=15) else 2, axis = 1)
df['dt_placement'] = df.apply(lambda row: f"{row['year']:04}-{row['month']:02}-{1 if row['period'] == 1 else 16:02}", axis = 1)

df['dt_placement']= pd.to_datetime(df['dt_placement'])
df['case'] = 1

In [9]:
df


,onset of symptoms,year,month,day,nuts2,nuts3,lau1,period,dt_placement,case
0,2008-08-29,2008,8,29,veneto,rovigo,ficarolo,2,2008-08-16,1
1,2009-08-20,2009,8,20,veneto,venezia,cona,2,2009-08-16,1
2,2009-08-23,2009,8,23,veneto,rovigo,villadose,2,2009-08-16,1
3,2009-09-04,2009,9,4,veneto,rovigo,fratta polesine,1,2009-09-01,1
4,2009-09-10,2009,9,10,veneto,rovigo,occhiobello,1,2009-09-01,1
...,...,...,...,...,...,...,...,...,...,...
293,2019-09-20,2019,9,20,veneto,venezia,saint maria da sala,2,2019-09-16,1
294,2019-09-26,2019,9,26,veneto,padova,vigodarzere,2,2019-09-16,1
295,2019-09-26,2019,9,26,veneto,venezia,caorle,2,2019-09-16,1
296,2019-10-05,2019,10,5,veneto,venezia,noventa da piave,1,2019-10-01,1


In [10]:
df.drop(columns = ['onset of symptoms', 'day', 'nuts3', 'period'], inplace = True)

In [11]:
df = df[['month', 'year', 'lau1', 'nuts2', 'dt_placement', 'case']]

In [12]:
df

,month,year,lau1,nuts2,dt_placement,case
0,8,2008,ficarolo,veneto,2008-08-16,1
1,8,2009,cona,veneto,2009-08-16,1
2,8,2009,villadose,veneto,2009-08-16,1
3,9,2009,fratta polesine,veneto,2009-09-01,1
4,9,2009,occhiobello,veneto,2009-09-01,1
...,...,...,...,...,...,...
293,9,2019,saint maria da sala,veneto,2019-09-16,1
294,9,2019,vigodarzere,veneto,2019-09-16,1
295,9,2019,caorle,veneto,2019-09-16,1
296,10,2019,noventa da piave,veneto,2019-10-01,1


In [13]:
case_months = df['month'].drop_duplicates().sort_values().tolist()
case_months

[6, 7, 8, 9, 10, 11]

In [14]:
case_years = df['year'].drop_duplicates().sort_values().tolist()
case_years

[2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

In [15]:
with open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_2008-2019_Cases_Dates.txt', 'w', encoding=enc) as f:

    f.write("%s: " % 'months')
    for item in case_months:
        f.write("%s, " % item)

    f.write("\n")

    f.write("%s: " % 'years')
    for item in case_years:
        f.write("%s, " % item)

In [16]:
## Creating DataFrame to hold negative examples

df_timeline = pd.DataFrame(columns = ['lau1', 'nuts2', 'dt_placement', 'case'])

for municipality in df['lau1'].drop_duplicates().sort_values():
    for year in range(2008, 2020):
        for month in range (6,12):
            df_timeline = df_timeline.append({'lau1' : municipality, 'nuts2' : NUTS2_LOWER, 'dt_placement' : f"{year:04}-{month:02}-{1:02}", 'case' : 0}, ignore_index = True)
            df_timeline = df_timeline.append({'lau1' : municipality, 'nuts2' : NUTS2_LOWER, 'dt_placement' : f"{year:04}-{month:02}-{16:02}", 'case' : 0}, ignore_index = True)
            
df_timeline 

,lau1,nuts2,dt_placement,case
0,adria,veneto,2008-06-01,0
1,adria,veneto,2008-06-16,0
2,adria,veneto,2008-07-01,0
3,adria,veneto,2008-07-16,0
4,adria,veneto,2008-08-01,0
...,...,...,...,...
25195,zevio,veneto,2019-09-16,0
25196,zevio,veneto,2019-10-01,0
25197,zevio,veneto,2019-10-16,0
25198,zevio,veneto,2019-11-01,0


In [17]:
df_timeline['dt_placement']= pd.to_datetime(df_timeline['dt_placement'])
df.reset_index(inplace = True, drop=True)
df_timeline.reset_index(inplace = True, drop=True)

In [18]:
## Add cases to negative examples DataFrame

#iterrate DataFrame with cases by row
for index, row in df.iterrows():
    #find index at DataFrame with negative examples where location and date matches the DataFrame with cases
    idx = df_timeline.index[(df_timeline['lau1']==row['lau1']) & (df_timeline['dt_placement']==row['dt_placement'])]
    #get the previous case number using index
    previous_case = df_timeline._get_value(idx.values[0], 'case')
    #add 1 (one) to column case
    df_timeline.at[idx.values[0],'case'] = previous_case+1

In [19]:
df_timeline.case.sum()

298

In [20]:
df_timeline['case'].value_counts().sort_index()

0    24925
1      258
2       15
4        1
6        1
Name: case, dtype: int64

In [21]:
df_timeline

,lau1,nuts2,dt_placement,case
0,adria,veneto,2008-06-01,0
1,adria,veneto,2008-06-16,0
2,adria,veneto,2008-07-01,0
3,adria,veneto,2008-07-16,0
4,adria,veneto,2008-08-01,0
...,...,...,...,...
25195,zevio,veneto,2019-09-16,0
25196,zevio,veneto,2019-10-01,0
25197,zevio,veneto,2019-10-16,0
25198,zevio,veneto,2019-11-01,0


In [22]:
df_timeline.to_csv(f"../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Cases_2008-2019.csv", encoding = enc, index = False)